In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("datasetCarga.csv")
df = pd.read_csv("datasetCarga.csv", encoding='latin1')

df.head()
df.info()
df.describe()

df.isnull().sum()
df = df.drop_duplicates()

C:\Users\Jhank\AppData\Local\Temp\ipykernel_23176\1115170545.py:4: DtypeWarning: Columns (0: COD_PRODUCTO, 1: CANTIDAD) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("datasetCarga.csv")
C:\Users\Jhank\AppData\Local\Temp\ipykernel_23176\1115170545.py:5: DtypeWarning: Columns (0: COD_PRODUCTO, 1: CANTIDAD) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("datasetCarga.csv", encoding='latin1')


<class 'pandas.DataFrame'>
RangeIndex: 836013 entries, 0 to 836012
Data columns (total 26 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   AÃO/MES                836013 non-null  int64  
 1   NATURALEZA              834351 non-null  str    
 2   COD_PRODUCTO            836013 non-null  object 
 3   PRODUCTO                836013 non-null  str    
 4   CANTIDAD                836013 non-null  object 
 5   UNID_MEDIDA             834351 non-null  str    
 6   FECHASALIDACARGUE       836013 non-null  str    
 7   HORA_SALIDA_CARGUE      836013 non-null  str    
 8   FECHALLEGADADESCARGUE   836013 non-null  str    
 9   HORA_LLEGADA_DESCARGUE  836013 non-null  str    
 10  CODIGO_CARGUE           836013 non-null  int64  
 11  CARGUE                  836013 non-null  str    
 12  CODIGO_DESCARGUE        836013 non-null  int64  
 13  DESCARGUE               835988 non-null  str    
 14  HORAS_VIAJE             836013 

In [15]:
#Cargar y Limpiar columnas
columnas_numericas = [
    "CANTIDAD",
    "HORAS_VIAJE",
    "HORAS_ESPERA_CARGUE",
    "HORAS_CARGUE",
    "HORAS_ESPERA_DESCARGUE",
    "HORAS_DESCARGUE",
    "VALOR_PACTADO",
    "VALOR_PAGADO",
    "CANTIDAD_REMESAS_VIAJE",
    "EMPRESA_TRANSPORTE",
    "PLACA",
    "CONFIGURACION",
    "CONDUCTOR"
]

for col in columnas_numericas:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )

    df[col] = pd.to_numeric(df[col], errors='coerce')

#Convertir fechas
df["FECHASALIDACARGUE"] = pd.to_datetime(
    df["FECHASALIDACARGUE"],
    errors='coerce'
)

df["FECHALLEGADADESCARGUE"] = pd.to_datetime(
    df["FECHALLEGADADESCARGUE"],
    errors='coerce'
)

C:\Users\Jhank\AppData\Local\Temp\ipykernel_23176\3140131436.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FECHASALIDACARGUE"] = pd.to_datetime(
C:\Users\Jhank\AppData\Local\Temp\ipykernel_23176\3140131436.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FECHALLEGADADESCARGUE"] = pd.to_datetime(


In [16]:
#Extraer las variables más importantes
df["MES"] = df["FECHASALIDACARGUE"].dt.month
df["DIA_SEMANA"] = df["FECHASALIDACARGUE"].dt.dayofweek

#Convertir horas para solo extraer lahora y no segundos
df["HORA_SALIDA_CARGUE"] = pd.to_datetime(
    df["HORA_SALIDA_CARGUE"],
    errors='coerce'
).dt.hour
df["HORA_LLEGADA_DESCARGUE"] = pd.to_datetime(
    df["HORA_LLEGADA_DESCARGUE"],
    errors='coerce'
).dt.hour

In [17]:
#Eliminar filas con datos vacíos
df = df.dropna(subset=["HORAS_VIAJE"])
df = df[df["PRODUCTO"] != "VARIOS CONTENEDOR VACIO"]

#eliminar columnas que no se usarán 
columnas_eliminar = [
    "CONFIGURACION",
    "CONDUCTOR",
    "PLACA",
    "EMPRESA_TRANSPORTE"
]

df = df.drop(columns=columnas_eliminar)

#Eliminar filas vacías en columna naturaleza
df["NATURALEZA"] = df["NATURALEZA"].astype(str).str.strip()
df["NATURALEZA"] = df["NATURALEZA"].replace("nan", np.nan)
df = df.dropna(subset=["NATURALEZA"])
df = df[df["NATURALEZA"] != ""]

df.info()
df.head()

#Guardar dataset limpio
df.to_csv("dataset_limpio.csv", index=False)

<class 'pandas.DataFrame'>
Index: 760464 entries, 0 to 836012
Data columns (total 24 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   AÃO/MES                760464 non-null  int64         
 1   NATURALEZA              760464 non-null  str           
 2   COD_PRODUCTO            760464 non-null  object        
 3   PRODUCTO                760464 non-null  str           
 4   CANTIDAD                760463 non-null  float64       
 5   UNID_MEDIDA             760464 non-null  str           
 6   FECHASALIDACARGUE       760464 non-null  datetime64[us]
 7   HORA_SALIDA_CARGUE      759891 non-null  float64       
 8   FECHALLEGADADESCARGUE   760464 non-null  datetime64[us]
 9   HORA_LLEGADA_DESCARGUE  759891 non-null  float64       
 10  CODIGO_CARGUE           760464 non-null  int64         
 11  CARGUE                  760464 non-null  str           
 12  CODIGO_DESCARGUE        760464 non-null  int64

In [19]:
# ==========================================
# MODELO PARA PREDECIR HORAS DE VIAJE
# ==========================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

# Copia dataset
tiempo_df = df.copy()

# Variables que NO deben usarse
tiempo_df = tiempo_df.drop(columns=[
    "HORAS_VIAJE",
    "VALOR_PAGADO",
    "VALOR_PACTADO",
    "FECHASALIDACARGUE",
    "FECHALLEGADADESCARGUE"
], errors="ignore")

# Encoding
le_dict = {}

for col in tiempo_df.select_dtypes(include='object').columns:

    le = LabelEncoder()

    tiempo_df[col] = le.fit_transform(
        tiempo_df[col].astype(str)
    )

    le_dict[col] = le

# Variables
X = tiempo_df

y = df["HORAS_VIAJE"]

# División
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Modelo
modelo_tiempo = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Entrenamiento
modelo_tiempo.fit(X_train, y_train)

# Predicción
predicciones = modelo_tiempo.predict(X_test)

# Evaluación
r2 = r2_score(y_test, predicciones)

print("R²:", r2)

C:\Users\Jhank\AppData\Local\Temp\ipykernel_23176\427602635.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in tiempo_df.select_dtypes(include='object').columns:


R²: 0.7231632234948748


In [20]:
# ==========================================
# MODELO PARA PREDECIR COSTO DEL VIAJE
# ==========================================

costo_df = df.copy()

# Eliminar columnas no válidas
costo_df = costo_df.drop(columns=[
    "VALOR_PAGADO",
    "VALOR_PACTADO",
    "FECHASALIDACARGUE",
    "FECHALLEGADADESCARGUE"
], errors="ignore")

# Encoding
for col in costo_df.select_dtypes(include='object').columns:

    le = LabelEncoder()

    costo_df[col] = le.fit_transform(
        costo_df[col].astype(str)
    )

# Variables
X = costo_df

y = df["VALOR_PAGADO"]

# División
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Modelo
modelo_costo = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Entrenamiento
modelo_costo.fit(X_train, y_train)

# Predicción
predicciones = modelo_costo.predict(X_test)

# Resultado
print("R²:", r2_score(y_test, predicciones))

C:\Users\Jhank\AppData\Local\Temp\ipykernel_23176\3549275794.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in costo_df.select_dtypes(include='object').columns:


MemoryError: could not allocate 33554432 bytes